# MIE 446 — Code-to-Print Wing

**Fall 2026 | Team project | Python + CadQuery + 3D printing**

In this notebook your team will define a parametric semi-wing, check the analytical geometry, generate a printable CAD model, inspect it, and export traceable STL/3MF/STEP files. The wing is a **non-flying fabrication demonstrator**. Do not use it for flight, load testing, or any airworthiness claim.

## Learning and AI-use expectations

AI may be used for brainstorming, code generation, refactoring, debugging, tests, documentation, and critique. Your team remains responsible for every submitted line and claim. Record material AI help in the course log, validate it independently, and be ready to explain the code without AI during the individual defense.

Before changing a parameter, write a prediction. After execution, compare the result with your prediction and explain any mismatch.

In [ ]:
#@title 1. Install the pinned course environment
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Ehsan-Roohi/MIE446-Code-to-Print-Wing.git"
REPO_DIR = Path("/content/MIE446-Code-to-Print-Wing")

if not (REPO_DIR / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[dev]"],
    check=True,
)
sys.path.insert(0, str(REPO_DIR / "src"))
print("Course environment ready.")

In [ ]:
# 2. Installation smoke test
from importlib.metadata import version
from math import isclose, pi

import cadquery as cq

assert version("cadquery") == "2.8.0"
assert version("cadquery-ocp") == "7.9.3.1.1"
smoke = cq.Workplane("XY").box(20, 30, 4).faces(">Z").workplane().hole(6)
assert smoke.val().isValid()
assert isclose(smoke.val().Volume(), 20 * 30 * 4 - pi * 3**2 * 4, rel_tol=1e-6)
print("CadQuery", version("cadquery"), "and OCP", version("cadquery-ocp"), "passed the smoke test.")

## 3. Define the design

Edit the cell below. Keep the semi-span at 450 mm, use at most three modules, and select an instructor-approved four-digit NACA profile. Do not change spar diameter until the course-issued rods have been measured.

In [ ]:
#@title STUDENT DESIGN CELL — edit and justify these values
from mie446_wing import SparSpec, WingParameters

TEAM_ID = "Team00"  #@param {type:"string"}
REVISION = "R01"  #@param {type:"string"}
NACA_CODE = "2412"  #@param {type:"string"}
ROOT_CHORD_MM = 160.0  #@param {type:"number"}
TIP_CHORD_MM = 100.0  #@param {type:"number"}
SKIN_MM = 1.2  #@param {type:"number"}
RIB_THICKNESS_MM = 1.6  #@param {type:"number"}
MODULE_COUNT = 3  #@param {type:"slider", min:1, max:3, step:1}
SELECTED_RADIAL_CLEARANCE_MM = 0.25  # replace only after printing the fit coupon

parameters = WingParameters(
    naca=NACA_CODE,
    semi_span_mm=450.0,
    root_chord_mm=ROOT_CHORD_MM,
    tip_chord_mm=TIP_CHORD_MM,
    skin_mm=SKIN_MM,
    rib_thickness_mm=RIB_THICKNESS_MM,
    module_count=MODULE_COUNT,
    spars=(
        SparSpec(0.30, rod_diameter_mm=4.0, radial_clearance_mm=SELECTED_RADIAL_CLEARANCE_MM),
        SparSpec(0.60, rod_diameter_mm=4.0, radial_clearance_mm=SELECTED_RADIAL_CLEARANCE_MM),
    ),
)
parameters

In [ ]:
# 4. Independent analytical checks — run these before CAD
from pprint import pprint
from mie446_wing import calculate_planform_metrics, validate_wing

metrics = calculate_planform_metrics(parameters)
pprint(metrics.to_dict())

parameter_report = validate_wing(parameters=parameters)
print(parameter_report.summary())
parameter_report.raise_for_failure()

## 5. Controlled-change prediction

Before running the next cell, write your prediction in the strings below. A useful prediction states the direction of change and why. The comparison uses analytical values, not an AI answer.

In [ ]:
from dataclasses import replace

PREDICTION_AREA = "Write whether full-wing area will increase, decrease, or stay constant, and why."
PREDICTION_AR = "Write whether aspect ratio will increase, decrease, or stay constant, and why."

comparison_parameters = replace(parameters, tip_chord_mm=parameters.tip_chord_mm + 10.0)
comparison = calculate_planform_metrics(comparison_parameters)
print(PREDICTION_AREA)
print("Computed full-area change (mm^2):", comparison.equivalent_full_area_mm2 - metrics.equivalent_full_area_mm2)
print(PREDICTION_AR)
print("Computed aspect-ratio change:", comparison.aspect_ratio - metrics.aspect_ratio)
print("Explain whether the output agrees with your prediction before continuing.")

In [ ]:
# 6. Run fast unit tests before building geometry
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_airfoil.py", "tests/test_metrics.py", "tests/test_parameters.py"],
    cwd=REPO_DIR,
)
assert result.returncode == 0, "A fast unit test failed. Diagnose it before building CAD."

In [ ]:
# 7. Build the CadQuery model
from time import perf_counter
from mie446_wing import build_wing

started = perf_counter()
build = build_wing(parameters)
print(f"Built {len(build.modules)} printable modules in {perf_counter() - started:.1f} s.")
print(f"Final CAD material volume: {build.complete.Volume():,.1f} mm^3")

In [ ]:
# 8. Interactive browser preview
from mie446_wing import plot_shape

figure = plot_shape(build.complete, title=f"MIE 446 {TEAM_ID} — complete semi-wing")
figure.show()

In [ ]:
# 9. Full geometry validation
report = validate_wing(build)
print(report.summary())
report.raise_for_failure()
print("Code checks passed. Slicer inspection and physical measurement are still required.")

## 10. Required engineering interpretation

Before export, your team should be able to answer:

1. How does `tip_chord_mm` propagate from the parameter cell to the final STL?
2. Why is the inner cavity an approximation rather than an exact constant-normal-thickness shell?
3. Which check would detect a disconnected module?
4. Why are ribs placed on both sides of a module seam instead of exactly on the cut plane?
5. Which print defects cannot be detected by this code?

In [ ]:
# 11. Short AI Use and Validation Log — add one dictionary per material use
AI_LOG = [
    {
        "tool": "Example: ChatGPT",
        "purpose": "Example: help diagnose a failed geometry check",
        "affected_code_or_claim": "Example: student design cell / spar clearance",
        "student_change": "Replace with what your team changed",
        "independent_check": "Replace with a test, equation, documentation, or measurement",
        "error_or_limitation_found": "Replace with what AI missed or got wrong",
        "verdict": "Accept with Limitations",
    }
]

In [ ]:
# 12. Export, archive, and download
import json
from shutil import make_archive
from mie446_wing import export_build

OUTPUT_DIR = Path("/content") / f"MIE446_{TEAM_ID}_{REVISION}"
manifest = export_build(build, OUTPUT_DIR, team=TEAM_ID, revision=REVISION)
(OUTPUT_DIR / "ai_use_log.json").write_text(json.dumps(AI_LOG, indent=2), encoding="utf-8")
archive_path = make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Export complete:", archive_path)
print("Files in manifest:", len(manifest["files"]))

try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print("Not running in Colab; archive remains at", archive_path)

## After Colab

Import each module into the current staff-approved Creality K2 Pro / 0.4 mm / regular PLA Pro profile. Confirm millimetres and 100% scale, inspect every layer, print the fit coupon before design freeze, record the selected clearance, supervise the first layers, and measure the completed parts. Passing this notebook does not certify printability, structural performance, or flight safety.